In [1]:
import gdown

# ID-ul fisierului din link
file_id: str = "11qXWGybF825aTTVAidBuUuAfD4hcGRoF"

# URL generat automat pentru download
url: str = f"https://drive.google.com/uc?id={file_id}"

# Descarca fisierul
gdown.download(url, "recenzii_100.csv", quiet=False)

print("Download complet!")

Downloading...
From: https://drive.google.com/uc?id=11qXWGybF825aTTVAidBuUuAfD4hcGRoF
To: /content/recenzii_100.csv
100%|██████████| 10.6k/10.6k [00:00<00:00, 20.7MB/s]

Download complet!


In [2]:
import pandas as pd

In [3]:
pd.read_csv("recenzii_100.csv").head(10)

,text,label
0,Îmi place foarte mult aplicația pentru că ofe...,Pozitiv
1,Este una dintre cele mai bune aplicații deoare...,Pozitiv
2,Nu sunt mulțumit de aplicație deoarece oferă ...,Negativ
3,Cred că aplicația s-ar îmbunătăți dacă ar perm...,Sugestie
4,Ar fi foarte util dacă aplicația ar putea fun...,Sugestie
5,Mi-ar plăcea să existe o funcționalitate care ...,Sugestie
6,Nu sunt mulțumit de aplicație deoarece permit...,Negativ
7,Cred că aplicația s-ar îmbunătăți dacă ar perm...,Sugestie
8,Este una dintre cele mai bune aplicații deoare...,Pozitiv
9,Nu recomand aplicația deoarece oferă o experi...,Negativ


In [ ]:
open_ai_k = 'sk-proj-loAqTIC9IPbtqPkAA5gSQW7_yUW6mQ-k_uBVFAomjEOVte2HgRnuQlvEa6eeDzAyQjw-MaQLXOT3BlbkFJHUBPBr_c9RsuVlSkcRbo371exKFvg4AmcsfEZmbwitMQVptPLNH5ViS72zFZRrjITM'

In [ ]:
open_ai_k += 'LastFewChars'

In [ ]:
from openai import OpenAI
import pandas as pd
from tqdm import tqdm


client = OpenAI(api_key=open_ai_k)

df = pd.read_csv("recenzii_100.csv")

few_shot = """
Clasifică următoarea recenzie într-una dintre categoriile: Pozitiv, Negativ, Sugestie.

Exemple:
Recenzie: "Îmi place aplicația, funcționează foarte bine și este ușor de folosit." → Etichetă: Pozitiv
Recenzie: "Se blochează frecvent și interfața e greu de folosit." → Etichetă: Negativ
Recenzie: "Mi-ar plăcea să existe o funcție de export în PDF." → Etichetă: Sugestie
"""


def clasifica(text):
    prompt = f"""{few_shot}

Recenzie: "{text}"
Etichetă:"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Eroare: {e}"

tqdm.pandas()
df["predictie"] = df["text"].progress_apply(clasifica)


100%|██████████| 100/100 [00:46<00:00,  2.17it/s]


In [ ]:
df[['label','predictie']].value_counts()

label     predictie         
Sugestie  Sugestie              32
Negativ   Negativ               26
Pozitiv   Pozitiv               19
Negativ   Etichetă: Negativ      8
Pozitiv   Negativ                6
Sugestie  Etichetă: Negativ      2
          Etichetă: Sugestie     2
Negativ   Sugestie               1
Pozitiv   Etichetă: Negativ      1
          Etichetă: Pozitiv      1
          Sugestie               1
Sugestie  Negativ                1
Name: count, dtype: int64